In [12]:
import pandas as pd
import numpy as np
from geopy.distance import geodesic
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# ─────────────────────────────────────────────
# 1. FACTORY & PRODUCT MAPPINGS
# ─────────────────────────────────────────────
FACTORIES = {
    "Lot's O' Nuts":    (32.881893, -111.768036),
    "Wicked Choccy's":  (32.076176, -81.088371),
    "Sugar Shack":      (48.11914,  -96.18115),
    "Secret Factory":   (41.446333, -90.565487),
    "The Other Factory":(35.1175,   -89.971107),
}

PRODUCT_FACTORY = {
    "Wonka Bar - Nutty Crunch Surprise":   "Lot's O' Nuts",
    "Wonka Bar - Fudge Mallows":           "Lot's O' Nuts",
    "Wonka Bar -Scrumdiddlyumptious":      "Lot's O' Nuts",
    "Wonka Bar - Milk Chocolate":          "Wicked Choccy's",
    "Wonka Bar - Triple Dazzle Caramel":   "Wicked Choccy's",
    "Laffy Taffy":                         "Sugar Shack",
    "SweeTARTS":                           "Sugar Shack",
    "Nerds":                               "Sugar Shack",
    "Fun Dip":                             "Sugar Shack",
    "Fizzy Lifting Drinks":                "Sugar Shack",
    "Everlasting Gobstopper":              "Secret Factory",
    "Hair Toffee":                         "The Other Factory",
    "Lickable Wallpaper":                  "Secret Factory",
    "Wonka Gum":                           "Secret Factory",
    "Kazookles":                           "The Other Factory",
}

REGION_COORDS = {
    "Atlantic": (35.0, -78.0),
    "Gulf":     (30.0, -90.0),
    "Interior": (41.0, -95.0),
    "Pacific":  (37.0, -120.0),
}

# ─────────────────────────────────────────────
# 2. LOAD DATA
# ─────────────────────────────────────────────
print("=" * 60)
print("STEP 1: LOADING DATA")
print("=" * 60)
df = pd.read_csv(r"C:\Users\Jackson Danie M D\Downloads\Nassau Candy Distributor.csv")
print(f"Loaded: {df.shape[0]} rows × {df.shape[1]} columns")

# ─────────────────────────────────────────────
# 3. DATE PARSING & LEAD TIME ENGINEERING
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 2: LEAD TIME FEATURE ENGINEERING")
print("=" * 60)

df["Order Date"] = pd.to_datetime(df["Order Date"], format="%d-%m-%Y")
df["Ship Date"]  = pd.to_datetime(df["Ship Date"],  format="%d-%m-%Y")

# Raw lead time in days
df["Lead Time Raw"] = (df["Ship Date"] - df["Order Date"]).dt.days

# Normalize: subtract minimum so lead time starts at 0
df["Lead Time Days"] = df["Lead Time Raw"] - df["Lead Time Raw"].min()

# Order month and year (seasonality features)
df["Order Month"] = df["Order Date"].dt.month
df["Order Year"]  = df["Order Date"].dt.year
df["Order DayOfWeek"] = df["Order Date"].dt.dayofweek

print(f"Lead Time Days — Min: {df['Lead Time Days'].min()}, "
      f"Max: {df['Lead Time Days'].max()}, "
      f"Mean: {df['Lead Time Days'].mean():.1f}")

# ─────────────────────────────────────────────
# 4. FACTORY ASSIGNMENT & DISTANCE FEATURE
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 3: FACTORY ASSIGNMENT & DISTANCE CALCULATION")
print("=" * 60)

df["Factory"] = df["Product Name"].map(PRODUCT_FACTORY)

def compute_distance(row):
    factory_coord = FACTORIES.get(row["Factory"])
    region_coord  = REGION_COORDS.get(row["Region"])
    if factory_coord and region_coord:
        return round(geodesic(factory_coord, region_coord).km, 2)
    return np.nan

df["Distance_km"] = df.apply(compute_distance, axis=1)

print("Distance_km stats:")
print(df["Distance_km"].describe())
print("\nAvg Distance by Factory:")
print(df.groupby("Factory")["Distance_km"].mean().sort_values())

# ─────────────────────────────────────────────
# 5. PROFIT MARGIN FEATURE
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 4: DERIVED FINANCIAL FEATURES")
print("=" * 60)

df["Profit Margin %"]   = (df["Gross Profit"] / df["Sales"] * 100).round(2)
df["Revenue Per Unit"]  = (df["Sales"] / df["Units"]).round(2)
df["Cost Per Unit"]     = (df["Cost"] / df["Units"]).round(2)
df["Profit Per Unit"]   = (df["Gross Profit"] / df["Units"]).round(2)

print("Profit Margin % stats:")
print(df["Profit Margin %"].describe())

# ─────────────────────────────────────────────
# 6. OUTLIER REMOVAL
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 5: OUTLIER REMOVAL")
print("=" * 60)

before = len(df)
numeric_cols = ["Sales", "Gross Profit", "Cost", "Units", "Distance_km"]
z_scores = np.abs(stats.zscore(df[numeric_cols]))
df = df[(z_scores < 3).all(axis=1)].reset_index(drop=True)
after = len(df)
print(f"Rows before: {before} | After outlier removal: {after} | Removed: {before - after}")

# ─────────────────────────────────────────────
# 7. ENCODING CATEGORICAL VARIABLES
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 6: ENCODING CATEGORICAL VARIABLES")
print("=" * 60)

le_dict = {}
cat_cols = ["Ship Mode", "Region", "Division", "Factory", "Product Name"]
for col in cat_cols:
    le = LabelEncoder()
    df[f"{col}_Enc"] = le.fit_transform(df[col])
    le_dict[col] = le
    print(f"{col}: {list(le.classes_)}")

# ─────────────────────────────────────────────
# 8. FEATURE MATRIX
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 7: BUILDING FEATURE MATRIX")
print("=" * 60)

FEATURE_COLS = [
    "Ship Mode_Enc",
    "Region_Enc",
    "Division_Enc",
    "Factory_Enc",
    "Product Name_Enc",
    "Distance_km",
    "Sales",
    "Units",
    "Cost",
    "Profit Margin %",
    "Revenue Per Unit",
    "Cost Per Unit",
    "Profit Per Unit",
    "Order Month",
    "Order DayOfWeek",
]

TARGET_COL = "Lead Time Days"

X = df[FEATURE_COLS].copy()
y = df[TARGET_COL].copy()

# Normalize features
scaler = MinMaxScaler()
X_scaled = pd.DataFrame(
    scaler.fit_transform(X),
    columns=FEATURE_COLS
)

print(f"Feature matrix shape: {X_scaled.shape}")
print(f"Target variable shape: {y.shape}")
print(f"\nFeatures used: {FEATURE_COLS}")

# ─────────────────────────────────────────────
# 9. SAVE OUTPUTS
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 8: SAVING OUTPUTS")
print("=" * 60)

import pickle, os
os.makedirs("/home/outputs", exist_ok=True)

df.to_csv("/home/outputs/nassau_engineered.csv", index=False)
X_scaled.to_csv("/home/outputs/X_scaled.csv", index=False)
y.to_csv("/home/outputs/y_target.csv", index=False)

with open("/home/outputs/label_encoders.pkl", "wb") as f:
    pickle.dump(le_dict, f)
with open("/home/outputs/scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)
with open("/home/outputs/feature_cols.pkl", "wb") as f:
    pickle.dump(FEATURE_COLS, f)

print("Saved: nassau_engineered.csv")
print("Saved: X_scaled.csv")
print("Saved: y_target.csv")
print("Saved: label_encoders.pkl")
print("Saved: scaler.pkl")
print("Saved: feature_cols.pkl")
print("\n✅ EDA & Feature Engineering Complete!")


STEP 1: LOADING DATA
Loaded: 10194 rows × 18 columns

STEP 2: LEAD TIME FEATURE ENGINEERING
Lead Time Days — Min: 0, Max: 738, Mean: 416.8

STEP 3: FACTORY ASSIGNMENT & DISTANCE CALCULATION
Distance_km stats:
count    10194.000000
mean      1853.687233
std       1078.139044
min        375.080000
25%        879.400000
50%       1739.810000
75%       3115.850000
max       3588.400000
Name: Distance_km, dtype: float64

Avg Distance by Factory:
Factory
The Other Factory    1473.994600
Secret Factory       1477.232120
Wicked Choccy's      1788.648271
Sugar Shack          1795.852727
Lot's O' Nuts        1922.487361
Name: Distance_km, dtype: float64

STEP 4: DERIVED FINANCIAL FEATURES
Profit Margin % stats:
count    10194.000000
mean        66.513003
std          6.721503
min          7.690000
25%         65.330000
50%         66.670000
75%         69.440000
max         80.000000
Name: Profit Margin %, dtype: float64

STEP 5: OUTLIER REMOVAL
Rows before: 10194 | After outlier removal: 9974 |

In [13]:
import pandas as pd
import numpy as np
import pickle, os, warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, RandomForestClassifier
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, accuracy_score, classification_report
from xgboost import XGBRegressor, XGBClassifier

# ─────────────────────────────────────────────
# LOAD
# ─────────────────────────────────────────────
print("=" * 60)
print("STEP 1: LOADING DATA & CREATING LEAD TIME TIERS")
print("=" * 60)

df = pd.read_csv("/home/outputs/nassau_engineered.csv")
X  = pd.read_csv("/home/outputs/X_scaled.csv")
y  = pd.read_csv("/home/outputs/y_target.csv").squeeze()

# Create lead time tier (3 natural clusters)
def tier(val):
    if val <= 15:   return 0   # Fast  (~0-15 days relative)
    elif val <= 400: return 1  # Medium
    else:            return 2  # Slow

y_tier = y.apply(tier)
tier_labels = {0: "Fast", 1: "Medium", 2: "Slow"}
print("Lead Time Tier distribution:")
print(y_tier.value_counts().map(lambda x: f"{x} rows").rename(tier_labels))

# ─────────────────────────────────────────────
# REGRESSION — predict exact lead time
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 2: REGRESSION MODELS (predict lead time days)")
print("=" * 60)

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)

reg_models = {
    "Linear Regression":  LinearRegression(),
    "Random Forest":      RandomForestRegressor(n_estimators=200, max_depth=10, random_state=42, n_jobs=-1),
    "Gradient Boosting":  GradientBoostingRegressor(n_estimators=200, max_depth=5, learning_rate=0.05, random_state=42),
    "XGBoost":            XGBRegressor(n_estimators=200, max_depth=5, learning_rate=0.05, random_state=42, verbosity=0),
}

reg_results = {}
for name, m in reg_models.items():
    m.fit(X_tr, y_tr)
    yp = m.predict(X_te)
    rmse = np.sqrt(mean_squared_error(y_te, yp))
    mae  = mean_absolute_error(y_te, yp)
    r2   = r2_score(y_te, yp)
    reg_results[name] = {"model": m, "RMSE": round(rmse,2), "MAE": round(mae,2), "R2": round(r2,4)}
    print(f"  {name:25s} | RMSE: {rmse:7.2f} | MAE: {mae:7.2f} | R²: {r2:.4f}")

# ─────────────────────────────────────────────
# CLASSIFICATION — predict lead time tier
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 3: CLASSIFICATION MODELS (predict Fast/Medium/Slow)")
print("=" * 60)

X_tr2, X_te2, y_tr2, y_te2 = train_test_split(X, y_tier, test_size=0.2, random_state=42)

clf_models = {
    "Random Forest Classifier": RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42, n_jobs=-1),
    "XGBoost Classifier":       XGBClassifier(n_estimators=200, max_depth=5, learning_rate=0.05, random_state=42, verbosity=0, use_label_encoder=False, eval_metric='mlogloss'),
}

clf_results = {}
for name, m in clf_models.items():
    m.fit(X_tr2, y_tr2)
    yp = m.predict(X_te2)
    acc = accuracy_score(y_te2, yp)
    clf_results[name] = {"model": m, "Accuracy": round(acc, 4)}
    print(f"\n  {name}")
    print(f"  Accuracy: {acc:.4f}")
    print(classification_report(y_te2, yp, target_names=["Fast","Medium","Slow"]))

# ─────────────────────────────────────────────
# BEST MODELS
# ─────────────────────────────────────────────
print("=" * 60)
print("STEP 4: BEST MODEL SELECTION")
print("=" * 60)

best_reg_name = max(reg_results, key=lambda k: reg_results[k]["R2"])
best_clf_name = max(clf_results, key=lambda k: clf_results[k]["Accuracy"])
best_reg  = reg_results[best_reg_name]["model"]
best_clf  = clf_results[best_clf_name]["model"]

print(f"🏆 Best Regressor  : {best_reg_name}  | R²={reg_results[best_reg_name]['R2']}  RMSE={reg_results[best_reg_name]['RMSE']}")
print(f"🏆 Best Classifier : {best_clf_name} | Accuracy={clf_results[best_clf_name]['Accuracy']}")

# ─────────────────────────────────────────────
# FEATURE IMPORTANCE
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 5: FEATURE IMPORTANCE (Best Regressor - Random Forest)")
print("=" * 60)

fi = pd.Series(best_reg.feature_importances_, index=X.columns).sort_values(ascending=False)
print(fi.to_string())

# ─────────────────────────────────────────────
# SAVE
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 6: SAVING ALL OUTPUTS")
print("=" * 60)

os.makedirs("/home/outputs", exist_ok=True)

for name, r in reg_results.items():
    fname = name.lower().replace(" ", "_")
    with open(f"/home/outputs/reg_{fname}.pkl", "wb") as f:
        pickle.dump(r["model"], f)

for name, r in clf_results.items():
    fname = name.lower().replace(" ", "_")
    with open(f"/home/outputs/clf_{fname}.pkl", "wb") as f:
        pickle.dump(r["model"], f)

with open("/home/outputs/best_model.pkl", "wb") as f:
    pickle.dump(best_reg, f)
with open("/home/outputs/best_model_name.pkl", "wb") as f:
    pickle.dump(best_reg_name, f)
with open("/home/outputs/best_classifier.pkl", "wb") as f:
    pickle.dump(best_clf, f)

# Save comparison tables
reg_df = pd.DataFrame([
    {"Model": n, "RMSE": v["RMSE"], "MAE": v["MAE"], "R2": v["R2"]}
    for n, v in reg_results.items()
]).sort_values("R2", ascending=False)

clf_df = pd.DataFrame([
    {"Model": n, "Accuracy": v["Accuracy"]}
    for n, v in clf_results.items()
]).sort_values("Accuracy", ascending=False)

reg_df.to_csv("/home/outputs/regression_comparison.csv", index=False)
clf_df.to_csv("/home/outputs/classification_comparison.csv", index=False)

# Save feature importance
fi_df = fi.reset_index()
fi_df.columns = ["Feature", "Importance"]
fi_df.to_csv("/home/outputs/feature_importance.csv", index=False)

# Save tier labels
with open("/home/outputs/tier_labels.pkl", "wb") as f:
    pickle.dump(tier_labels, f)

y_tier.to_csv("/home/outputs/y_tier.csv", index=False)

print("✅ All models saved!")
print("\nRegression Results:")
print(reg_df.to_string(index=False))
print("\nClassification Results:")
print(clf_df.to_string(index=False))

STEP 1: LOADING DATA & CREATING LEAD TIME TIERS
Lead Time Tier distribution:
Lead Time Days
Medium    4675 rows
Slow      3301 rows
Fast      1998 rows
Name: count, dtype: object

STEP 2: REGRESSION MODELS (predict lead time days)
  Linear Regression         | RMSE:  259.70 | MAE:  207.68 | R²: 0.0130
  Random Forest             | RMSE:  252.70 | MAE:  206.06 | R²: 0.0655
  Gradient Boosting         | RMSE:  254.93 | MAE:  207.49 | R²: 0.0489
  XGBoost                   | RMSE:  253.92 | MAE:  206.91 | R²: 0.0565

STEP 3: CLASSIFICATION MODELS (predict Fast/Medium/Slow)

  Random Forest Classifier
  Accuracy: 0.5033
              precision    recall  f1-score   support

        Fast       0.41      0.07      0.12       411
      Medium       0.52      0.87      0.66       949
        Slow       0.43      0.23      0.30       635

    accuracy                           0.50      1995
   macro avg       0.45      0.39      0.36      1995
weighted avg       0.47      0.50      0.43      1

In [14]:
import pandas as pd
import numpy as np
import pickle, os, warnings
warnings.filterwarnings('ignore')

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
from sklearn.impute import SimpleImputer

# ─────────────────────────────────────────────
# 1. LOAD DATA
# ─────────────────────────────────────────────
print("=" * 60)
print("STEP 1: LOADING ENGINEERED DATA")
print("=" * 60)

df = pd.read_csv("/home/outputs/nassau_engineered.csv")
print(f"Loaded: {df.shape[0]} rows × {df.shape[1]} columns")

# ─────────────────────────────────────────────
# 2. ROUTE CLUSTERING
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 2: ROUTE CLUSTERING")
print("=" * 60)

route_features = [
    "Lead Time Days", "Distance_km", "Profit Margin %",
    "Sales", "Gross Profit", "Units",
    "Ship Mode_Enc", "Region_Enc", "Factory_Enc",
]

route_df = df[route_features].copy()
imputer_route = SimpleImputer(strategy="median")
route_imputed = imputer_route.fit_transform(route_df)

scaler_route = StandardScaler()
route_scaled  = scaler_route.fit_transform(route_imputed)

print("Finding optimal K for Route Clustering...")
sil_scores = {}
for k in range(2, 8):
    km     = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(route_scaled)
    score  = silhouette_score(route_scaled, labels)
    sil_scores[k] = round(score, 4)
    print(f"  K={k} | Silhouette Score: {score:.4f}")

best_k_route = max(sil_scores, key=sil_scores.get)
print(f"\n✅ Best K for Routes: {best_k_route} (score={sil_scores[best_k_route]})")

km_route = KMeans(n_clusters=best_k_route, random_state=42, n_init=10)
df["Route_Cluster"] = km_route.fit_predict(route_scaled)

route_profile = df.groupby("Route_Cluster").agg(
    Count         = ("Route_Cluster", "count"),
    Avg_LeadTime  = ("Lead Time Days", "mean"),
    Avg_Distance  = ("Distance_km", "mean"),
    Avg_ProfitPct = ("Profit Margin %", "mean"),
    Avg_Sales     = ("Sales", "mean"),
).round(2)
print("\nRoute Cluster Profiles:")
print(route_profile.to_string())

# Label by lead time rank
lt_means = df.groupby("Route_Cluster")["Lead Time Days"].mean().sort_values()
cluster_labels = {}
n = len(lt_means)
for i, c in enumerate(lt_means.index):
    if i == 0:
        cluster_labels[c] = "Fast Route"
    elif i == n - 1:
        cluster_labels[c] = "Slow Route"
    else:
        cluster_labels[c] = f"Medium Route {i}"
df["Route_Label"] = df["Route_Cluster"].map(cluster_labels)

print("\nRoute Label Distribution:")
print(df["Route_Label"].value_counts().to_string())

# ─────────────────────────────────────────────
# 3. PRODUCT CLUSTERING
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 3: PRODUCT CLUSTERING")
print("=" * 60)

product_agg = df.groupby("Product Name").agg(
    Avg_LeadTime    = ("Lead Time Days", "mean"),
    Std_LeadTime    = ("Lead Time Days", "std"),
    Avg_Distance    = ("Distance_km", "mean"),
    Avg_ProfitPct   = ("Profit Margin %", "mean"),
    Avg_Sales       = ("Sales", "mean"),
    Avg_GrossProfit = ("Gross Profit", "mean"),
    Avg_Units       = ("Units", "mean"),
    Total_Orders    = ("Product Name", "count"),
).round(2)

prod_feat_cols = ["Avg_LeadTime", "Std_LeadTime", "Avg_Distance",
                  "Avg_ProfitPct", "Avg_Sales", "Avg_GrossProfit"]

# Impute NaN (e.g. Std_LeadTime is NaN for single-order products)
imputer_prod = SimpleImputer(strategy="median")
prod_imputed  = imputer_prod.fit_transform(product_agg[prod_feat_cols])

scaler_prod = StandardScaler()
prod_scaled  = scaler_prod.fit_transform(prod_imputed)

print("Finding optimal K for Product Clustering...")
sil_prod = {}
for k in range(2, 6):
    km     = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(prod_scaled)
    score  = silhouette_score(prod_scaled, labels)
    sil_prod[k] = round(score, 4)
    print(f"  K={k} | Silhouette Score: {score:.4f}")

best_k_prod = max(sil_prod, key=sil_prod.get)
print(f"\n✅ Best K for Products: {best_k_prod} (score={sil_prod[best_k_prod]})")

km_prod = KMeans(n_clusters=best_k_prod, random_state=42, n_init=10)
product_agg["Product_Cluster"] = km_prod.fit_predict(prod_scaled)

# Label product clusters
lt_prod   = product_agg.groupby("Product_Cluster")["Avg_LeadTime"].mean().sort_values()
prod_labels = {}
np_ = len(lt_prod)
for i, c in enumerate(lt_prod.index):
    if i == 0:
        prod_labels[c] = "High Performer"
    elif i == np_ - 1:
        prod_labels[c] = "Low Performer"
    else:
        prod_labels[c] = f"Mid Performer {i}"
product_agg["Product_Label"] = product_agg["Product_Cluster"].map(prod_labels)

print("\nProduct Cluster Assignments:")
print(product_agg[["Avg_LeadTime","Avg_Distance","Avg_ProfitPct",
                    "Total_Orders","Product_Cluster","Product_Label"]].to_string())

# ─────────────────────────────────────────────
# 4. SLOW & FAST ROUTE IDENTIFICATION
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 4: IDENTIFYING SLOW & FAST ROUTES")
print("=" * 60)

route_perf = df.groupby(["Factory", "Region", "Ship Mode"]).agg(
    Avg_LeadTime  = ("Lead Time Days", "mean"),
    Avg_Distance  = ("Distance_km", "mean"),
    Avg_ProfitPct = ("Profit Margin %", "mean"),
    Order_Count   = ("Lead Time Days", "count"),
).round(2).reset_index()

lt_75 = route_perf["Avg_LeadTime"].quantile(0.75)
lt_25 = route_perf["Avg_LeadTime"].quantile(0.25)

slow_routes = route_perf[route_perf["Avg_LeadTime"] >= lt_75].sort_values("Avg_LeadTime", ascending=False)
fast_routes = route_perf[route_perf["Avg_LeadTime"] <= lt_25].sort_values("Avg_LeadTime")

print(f"\nSlow Routes (Lead Time ≥ {lt_75:.0f} days):")
print(slow_routes.head(15).to_string(index=False))
print(f"\nFast Routes (Lead Time ≤ {lt_25:.0f} days):")
print(fast_routes.head(15).to_string(index=False))

# ─────────────────────────────────────────────
# 5. PRODUCT-REGION LEAD TIME MATRIX
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 5: PRODUCT × REGION LEAD TIME MATRIX")
print("=" * 60)

pivot = df.pivot_table(
    values="Lead Time Days",
    index="Product Name",
    columns="Region",
    aggfunc="mean"
).round(1)
print(pivot.to_string())

# ─────────────────────────────────────────────
# 6. FACTORY × REGION PERFORMANCE SUMMARY
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 6: FACTORY × REGION PERFORMANCE SUMMARY")
print("=" * 60)

fac_reg = df.groupby(["Factory", "Region"]).agg(
    Avg_LeadTime  = ("Lead Time Days", "mean"),
    Avg_Distance  = ("Distance_km", "mean"),
    Avg_ProfitPct = ("Profit Margin %", "mean"),
    Orders        = ("Lead Time Days", "count"),
).round(2)
print(fac_reg.to_string())

# ─────────────────────────────────────────────
# 7. SAVE ALL OUTPUTS
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 7: SAVING OUTPUTS")
print("=" * 60)

df.to_csv("/home/outputs/nassau_clustered.csv", index=False)
product_agg.to_csv("/home/outputs/product_clusters.csv")
route_perf.to_csv("/home/outputs/route_performance.csv", index=False)
slow_routes.to_csv("/home/outputs/slow_routes.csv", index=False)
fast_routes.to_csv("/home/outputs/fast_routes.csv", index=False)
pivot.to_csv("/home/outputs/product_region_matrix.csv")
fac_reg.to_csv("/home/outputs/factory_region_summary.csv")

for obj, fname in [
    (km_route,       "km_route.pkl"),
    (km_prod,        "km_product.pkl"),
    (scaler_route,   "scaler_route.pkl"),
    (cluster_labels, "cluster_labels.pkl"),
    (prod_labels,    "prod_labels.pkl"),
    (route_features, "route_features.pkl"),
    (imputer_route,  "imputer_route.pkl"),
    (imputer_prod,   "imputer_prod.pkl"),
]:
    with open(f"/home/outputs/{fname}", "wb") as f:
        pickle.dump(obj, f)
    print(f"Saved: {fname}")

print("\n✅ Route & Product Clustering Complete!")

STEP 1: LOADING ENGINEERED DATA
Loaded: 9974 rows × 34 columns

STEP 2: ROUTE CLUSTERING
Finding optimal K for Route Clustering...
  K=2 | Silhouette Score: 0.2444
  K=3 | Silhouette Score: 0.2007
  K=4 | Silhouette Score: 0.2409
  K=5 | Silhouette Score: 0.2146
  K=6 | Silhouette Score: 0.2604
  K=7 | Silhouette Score: 0.2373

✅ Best K for Routes: 6 (score=0.2604)

Route Cluster Profiles:
               Count  Avg_LeadTime  Avg_Distance  Avg_ProfitPct  Avg_Sales
Route_Cluster                                                             
0               1125        426.40       3559.77          64.77      10.55
1               2388        415.36        919.08          64.46       9.84
2               2101        421.36       2748.67          69.09      10.39
3               2571        413.98       1236.82          69.22      10.56
4               1709        414.62       1893.76          67.44      25.48
5                 80        364.36       1525.63           7.69      10.20

Route 

In [15]:
import pandas as pd
import numpy as np
import pickle, os, warnings
warnings.filterwarnings('ignore')

from geopy.distance import geodesic

# ─────────────────────────────────────────────
# CONSTANTS
# ─────────────────────────────────────────────
FACTORIES = {
    "Lot's O' Nuts":     (32.881893, -111.768036),
    "Wicked Choccy's":   (32.076176, -81.088371),
    "Sugar Shack":       (48.11914,  -96.18115),
    "Secret Factory":    (41.446333, -90.565487),
    "The Other Factory": (35.1175,   -89.971107),
}

REGION_COORDS = {
    "Atlantic": (35.0, -78.0),
    "Gulf":     (30.0, -90.0),
    "Interior": (41.0, -95.0),
    "Pacific":  (37.0, -120.0),
}

PRODUCT_FACTORY = {
    "Wonka Bar - Nutty Crunch Surprise":  "Lot's O' Nuts",
    "Wonka Bar - Fudge Mallows":          "Lot's O' Nuts",
    "Wonka Bar -Scrumdiddlyumptious":     "Lot's O' Nuts",
    "Wonka Bar - Milk Chocolate":         "Wicked Choccy's",
    "Wonka Bar - Triple Dazzle Caramel":  "Wicked Choccy's",
    "Laffy Taffy":                        "Sugar Shack",
    "SweeTARTS":                          "Sugar Shack",
    "Nerds":                              "Sugar Shack",
    "Fun Dip":                            "Sugar Shack",
    "Fizzy Lifting Drinks":               "Sugar Shack",
    "Everlasting Gobstopper":             "Secret Factory",
    "Hair Toffee":                        "The Other Factory",
    "Lickable Wallpaper":                 "Secret Factory",
    "Wonka Gum":                          "Secret Factory",
    "Kazookles":                          "The Other Factory",
}

ALL_FACTORIES = list(FACTORIES.keys())

# ─────────────────────────────────────────────
# 1. LOAD DATA & MODELS
# ─────────────────────────────────────────────
print("=" * 60)
print("STEP 1: LOADING DATA & MODELS")
print("=" * 60)

df         = pd.read_csv("/home/outputs/nassau_clustered.csv")
route_perf = pd.read_csv("/home/outputs/route_performance.csv")

with open("/home/outputs/best_model.pkl", "rb") as f:
    model = pickle.load(f)
with open("/home/outputs/scaler.pkl", "rb") as f:
    scaler = pickle.load(f)
with open("/home/outputs/feature_cols.pkl", "rb") as f:
    feature_cols = pickle.load(f)
with open("/home/outputs/label_encoders.pkl", "rb") as f:
    le_dict = pickle.load(f)

print(f"Loaded dataset: {df.shape}")
print(f"Best model: Random Forest Regressor")
print(f"Features: {len(feature_cols)}")

# ─────────────────────────────────────────────
# 2. BASELINE PERFORMANCE PER PRODUCT
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 2: BASELINE PERFORMANCE PER PRODUCT")
print("=" * 60)

baseline = df.groupby(["Product Name", "Factory"]).agg(
    Baseline_LeadTime = ("Lead Time Days", "mean"),
    Baseline_Profit   = ("Profit Margin %", "mean"),
    Baseline_Distance = ("Distance_km", "mean"),
    Baseline_Sales    = ("Sales", "mean"),
    Order_Count       = ("Product Name", "count"),
).round(2).reset_index()

print(baseline.to_string(index=False))

# ─────────────────────────────────────────────
# 3. HELPER: predict lead time for a scenario
# ─────────────────────────────────────────────
def get_distance(factory, region):
    return round(geodesic(FACTORIES[factory], REGION_COORDS[region]).km, 2)

def predict_lead_time(product, factory, region, ship_mode, df_ref, model, scaler, le_dict, feature_cols):
    """Build a feature row and predict lead time using the trained model."""
    # Get a reference row for this product to fill financial features
    ref = df_ref[df_ref["Product Name"] == product]
    if len(ref) == 0:
        return np.nan
    ref = ref.iloc[0]

    distance = get_distance(factory, region)

    # Encode categoricals safely
    def safe_encode(le, val):
        if val in le.classes_:
            return le.transform([val])[0]
        return 0

    row = {
        "Ship Mode_Enc":     safe_encode(le_dict["Ship Mode"], ship_mode),
        "Region_Enc":        safe_encode(le_dict["Region"], region),
        "Division_Enc":      safe_encode(le_dict["Division"], ref["Division"]),
        "Factory_Enc":       safe_encode(le_dict["Factory"], factory),
        "Product Name_Enc":  safe_encode(le_dict["Product Name"], product),
        "Distance_km":       distance,
        "Sales":             ref["Sales"],
        "Units":             ref["Units"],
        "Cost":              ref["Cost"],
        "Profit Margin %":   ref["Profit Margin %"],
        "Revenue Per Unit":  ref["Revenue Per Unit"],
        "Cost Per Unit":     ref["Cost Per Unit"],
        "Profit Per Unit":   ref["Profit Per Unit"],
        "Order Month":       ref["Order Month"],
        "Order DayOfWeek":   ref["Order DayOfWeek"],
    }

    X_row = pd.DataFrame([row])[feature_cols]
    X_scaled = scaler.transform(X_row)
    pred = model.predict(X_scaled)[0]
    return round(max(0, pred), 2)

# ─────────────────────────────────────────────
# 4. SIMULATE ALL FACTORY REASSIGNMENTS
# For each product × region × ship_mode,
# test every alternate factory and predict new lead time
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 3: SIMULATING ALL FACTORY REASSIGNMENTS")
print("=" * 60)

simulation_records = []

products   = df["Product Name"].unique()
regions    = df["Region"].unique()
ship_modes = df["Ship Mode"].unique()

for product in products:
    current_factory = PRODUCT_FACTORY[product]
    prod_df = df[df["Product Name"] == product]

    for region in regions:
        for ship_mode in ship_modes:
            # Baseline: current factory
            subset = prod_df[
                (prod_df["Region"] == region) &
                (prod_df["Ship Mode"] == ship_mode)
            ]
            if len(subset) == 0:
                continue

            baseline_lt   = subset["Lead Time Days"].mean()
            baseline_dist = get_distance(current_factory, region)
            baseline_prof = subset["Profit Margin %"].mean()

            # Simulate each alternate factory
            for alt_factory in ALL_FACTORIES:
                if alt_factory == current_factory:
                    continue

                pred_lt   = predict_lead_time(
                    product, alt_factory, region, ship_mode,
                    df, model, scaler, le_dict, feature_cols
                )
                alt_dist  = get_distance(alt_factory, region)
                lt_change = baseline_lt - pred_lt
                dist_change = baseline_dist - alt_dist

                simulation_records.append({
                    "Product":          product,
                    "Current_Factory":  current_factory,
                    "Alt_Factory":      alt_factory,
                    "Region":           region,
                    "Ship_Mode":        ship_mode,
                    "Baseline_LT":      round(baseline_lt, 2),
                    "Predicted_LT":     round(pred_lt, 2),
                    "LT_Reduction":     round(lt_change, 2),
                    "LT_Reduction_Pct": round(lt_change / baseline_lt * 100, 2) if baseline_lt > 0 else 0,
                    "Baseline_Dist_km": baseline_dist,
                    "Alt_Dist_km":      alt_dist,
                    "Dist_Reduction_km":round(dist_change, 2),
                    "Baseline_Profit":  round(baseline_prof, 2),
                    "Orders_In_Subset": len(subset),
                })

sim_df = pd.DataFrame(simulation_records)
print(f"Total simulation scenarios generated: {len(sim_df)}")
print(f"\nSample scenarios:")
print(sim_df.head(5).to_string(index=False))

# ─────────────────────────────────────────────
# 5. PROFIT SENSITIVITY ANALYSIS
# Estimate profit impact of each reassignment
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 4: PROFIT SENSITIVITY ANALYSIS")
print("=" * 60)

# Distance cost proxy: 0.05% profit margin per 100 km saved
COST_PER_100KM = 0.05

sim_df["Est_Profit_Impact"] = (
    sim_df["Dist_Reduction_km"] / 100 * COST_PER_100KM
).round(4)

sim_df["New_Est_Profit"] = (
    sim_df["Baseline_Profit"] + sim_df["Est_Profit_Impact"]
).round(2)

# Profit stability flag: if new profit < baseline, flag as risk
sim_df["Profit_Risk"] = sim_df["New_Est_Profit"] < sim_df["Baseline_Profit"]

print("Profit Risk Distribution:")
print(sim_df["Profit_Risk"].value_counts().rename({True: "At Risk", False: "Stable"}))

# ─────────────────────────────────────────────
# 6. SCENARIO CONFIDENCE SCORE
# Confidence = based on order volume + LT reduction magnitude
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 5: SCENARIO CONFIDENCE SCORING")
print("=" * 60)

max_orders = sim_df["Orders_In_Subset"].max()
max_lt_red = sim_df["LT_Reduction"].abs().max()

sim_df["Confidence_Score"] = (
    0.5 * (sim_df["Orders_In_Subset"] / max_orders) +
    0.5 * (sim_df["LT_Reduction"].clip(lower=0) / max_lt_red)
).round(4)

print(f"Confidence Score range: {sim_df['Confidence_Score'].min():.4f} – {sim_df['Confidence_Score'].max():.4f}")
print(f"Mean Confidence Score: {sim_df['Confidence_Score'].mean():.4f}")

# ─────────────────────────────────────────────
# 7. WHAT-IF SCENARIO: CURRENT vs RECOMMENDED
# Best reassignment per product per region
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 6: WHAT-IF — BEST REASSIGNMENT PER PRODUCT × REGION")
print("=" * 60)

best_reassignment = (
    sim_df[sim_df["LT_Reduction"] > 0]
    .sort_values("LT_Reduction", ascending=False)
    .groupby(["Product", "Region"])
    .first()
    .reset_index()
)

summary_cols = ["Product", "Region", "Current_Factory", "Alt_Factory",
                "Baseline_LT", "Predicted_LT", "LT_Reduction",
                "LT_Reduction_Pct", "Dist_Reduction_km", "Confidence_Score"]

print(best_reassignment[summary_cols].sort_values(
    "LT_Reduction_Pct", ascending=False
).head(20).to_string(index=False))

# ─────────────────────────────────────────────
# 8. TOP-10 GLOBAL BEST SCENARIOS
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 7: TOP-10 HIGHEST IMPACT SCENARIOS")
print("=" * 60)

top10 = (
    sim_df[sim_df["LT_Reduction"] > 0]
    .sort_values(["LT_Reduction_Pct", "Confidence_Score"], ascending=False)
    .head(10)
)
print(top10[summary_cols].to_string(index=False))

# ─────────────────────────────────────────────
# 9. SAVE
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 8: SAVING OUTPUTS")
print("=" * 60)

sim_df.to_csv("/home/outputs/simulation_results.csv", index=False)
best_reassignment.to_csv("/home/outputs/best_reassignments.csv", index=False)
top10.to_csv("/home/outputs/top10_scenarios.csv", index=False)
baseline.to_csv("/home/outputs/baseline_performance.csv", index=False)

print("Saved: simulation_results.csv")
print("Saved: best_reassignments.csv")
print("Saved: top10_scenarios.csv")
print("Saved: baseline_performance.csv")
print("\n✅ Scenario Simulation Engine Complete!")

STEP 1: LOADING DATA & MODELS
Loaded dataset: (9974, 36)
Best model: Random Forest Regressor
Features: 15

STEP 2: BASELINE PERFORMANCE PER PRODUCT
                     Product Name           Factory  Baseline_LeadTime  Baseline_Profit  Baseline_Distance  Baseline_Sales  Order_Count
           Everlasting Gobstopper    Secret Factory             737.00            80.00             375.08           30.00            1
             Fizzy Lifting Drinks       Sugar Shack             369.50            60.00            1692.96           13.12            6
                          Fun Dip       Sugar Shack             368.67            40.00            1656.48            4.00            3
                      Hair Toffee The Other Factory             551.25            77.78            1765.93           19.12            4
                        Kazookles The Other Factory             364.36             7.69            1525.63           10.20           80
                      Laffy Taffy   

In [16]:
import pandas as pd
import numpy as np
import pickle, os, warnings
warnings.filterwarnings('ignore')

# ─────────────────────────────────────────────
# 1. LOAD SIMULATION RESULTS
# ─────────────────────────────────────────────
print("=" * 60)
print("STEP 1: LOADING SIMULATION RESULTS")
print("=" * 60)

sim_df      = pd.read_csv("/home/outputs/simulation_results.csv")
baseline_df = pd.read_csv("/home/outputs/baseline_performance.csv")
route_perf  = pd.read_csv("/home/outputs/route_performance.csv")

print(f"Simulation scenarios : {len(sim_df)}")
print(f"Baseline products    : {len(baseline_df)}")
print(f"Route combinations   : {len(route_perf)}")

# ─────────────────────────────────────────────
# 2. SCORING FUNCTION
# Composite score = weighted sum of 3 KPIs
#   - Lead Time Reduction %    (40%)
#   - Distance Reduction       (30%)
#   - Profit Stability         (30%)
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 2: COMPOSITE OPTIMIZATION SCORING")
print("=" * 60)

# Keep only beneficial reassignments (LT_Reduction > 0)
good = sim_df[sim_df["LT_Reduction"] > 0].copy()

# Normalize each component to 0-1
def normalize(series):
    mn, mx = series.min(), series.max()
    if mx == mn:
        return pd.Series(0.5, index=series.index)
    return (series - mn) / (mx - mn)

good["Score_LT"]     = normalize(good["LT_Reduction_Pct"])
good["Score_Dist"]   = normalize(good["Dist_Reduction_km"])
good["Score_Profit"] = normalize(good["New_Est_Profit"])

# Default weights
W_LT, W_DIST, W_PROFIT = 0.40, 0.30, 0.30

good["Composite_Score"] = (
    W_LT     * good["Score_LT"]     +
    W_DIST   * good["Score_Dist"]   +
    W_PROFIT * good["Score_Profit"]
).round(4)

print(f"Scored {len(good)} beneficial scenarios")
print(f"Composite Score range: {good['Composite_Score'].min():.4f} – {good['Composite_Score'].max():.4f}")

# ─────────────────────────────────────────────
# 3. TOP-N FACTORY REASSIGNMENT RECOMMENDATIONS
# Best recommendation per product (globally)
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 3: TOP FACTORY REASSIGNMENT RECOMMENDATIONS")
print("=" * 60)

top_per_product = (
    good.sort_values("Composite_Score", ascending=False)
    .groupby("Product")
    .first()
    .reset_index()
)

rec_cols = [
    "Product", "Current_Factory", "Alt_Factory", "Region", "Ship_Mode",
    "Baseline_LT", "Predicted_LT", "LT_Reduction_Pct",
    "Dist_Reduction_km", "Baseline_Profit", "New_Est_Profit",
    "Composite_Score", "Confidence_Score"
]

print("\n🏆 TOP RECOMMENDATION PER PRODUCT:")
print(top_per_product[rec_cols].sort_values(
    "Composite_Score", ascending=False
).to_string(index=False))

# ─────────────────────────────────────────────
# 4. TOP-5 GLOBAL RECOMMENDATIONS
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 4: TOP-5 GLOBAL RECOMMENDATIONS")
print("=" * 60)

top5 = good.sort_values("Composite_Score", ascending=False).head(5)
print(top5[rec_cols].to_string(index=False))

# ─────────────────────────────────────────────
# 5. SPEED vs PROFIT PRIORITY SCENARIOS
# Priority Slider = 0 (profit first) to 1 (speed first)
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 5: PRIORITY SCENARIOS — SPEED vs PROFIT")
print("=" * 60)

def score_with_priority(df, speed_weight):
    profit_weight = 1 - speed_weight
    w_lt     = speed_weight  * 0.70
    w_dist   = speed_weight  * 0.30
    w_profit = profit_weight
    df = df.copy()
    df["Priority_Score"] = (
        w_lt     * df["Score_LT"]     +
        w_dist   * df["Score_Dist"]   +
        w_profit * df["Score_Profit"]
    ).round(4)
    return df.sort_values("Priority_Score", ascending=False)

for label, sw in [("Speed Priority (0.9)", 0.9),
                   ("Balanced (0.5)",       0.5),
                   ("Profit Priority (0.1)", 0.1)]:
    ranked = score_with_priority(good, sw)
    top3   = ranked.groupby("Product").first().reset_index().head(3)
    print(f"\n── {label} ──")
    print(top3[["Product", "Alt_Factory", "LT_Reduction_Pct",
                 "New_Est_Profit", "Priority_Score"]].to_string(index=False))

# ─────────────────────────────────────────────
# 6. RISK ASSESSMENT
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 6: RISK ASSESSMENT")
print("=" * 60)

# High risk = profit drops AND low confidence
sim_df["Risk_Level"] = "Low"
sim_df.loc[
    (sim_df["Profit_Risk"] == True) & (sim_df["Confidence_Score"] < 0.2),
    "Risk_Level"
] = "High"
sim_df.loc[
    (sim_df["Profit_Risk"] == True) & (sim_df["Confidence_Score"] >= 0.2),
    "Risk_Level"
] = "Medium"

risk_summary = sim_df.groupby("Risk_Level")["Product"].count().rename("Count")
print("\nRisk Level Distribution:")
print(risk_summary.to_string())

high_risk = sim_df[sim_df["Risk_Level"] == "High"][[
    "Product", "Current_Factory", "Alt_Factory", "Region",
    "LT_Reduction_Pct", "Baseline_Profit", "New_Est_Profit", "Confidence_Score"
]].sort_values("LT_Reduction_Pct", ascending=False)

print(f"\nHigh Risk Scenarios ({len(high_risk)}):")
print(high_risk.head(10).to_string(index=False))

# ─────────────────────────────────────────────
# 7. KPI SUMMARY TABLE
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 7: KPI SUMMARY")
print("=" * 60)

beneficial = sim_df[sim_df["LT_Reduction"] > 0]

kpi = {
    "Total Scenarios Evaluated":       len(sim_df),
    "Beneficial Reassignments":        len(beneficial),
    "Avg Lead Time Reduction (days)":  round(beneficial["LT_Reduction"].mean(), 2),
    "Max Lead Time Reduction (days)":  round(beneficial["LT_Reduction"].max(), 2),
    "Avg LT Reduction (%)":            round(beneficial["LT_Reduction_Pct"].mean(), 2),
    "Max LT Reduction (%)":            round(beneficial["LT_Reduction_Pct"].max(), 2),
    "Profit-Stable Scenarios":         int((sim_df["Profit_Risk"] == False).sum()),
    "High Risk Scenarios":             int((sim_df["Risk_Level"] == "High").sum()),
    "Avg Confidence Score":            round(beneficial["Confidence_Score"].mean(), 4),
    "Products with Recommendations":   beneficial["Product"].nunique(),
    "Recommendation Coverage (%)":     round(beneficial["Product"].nunique() / sim_df["Product"].nunique() * 100, 2),
}

kpi_df = pd.DataFrame(list(kpi.items()), columns=["KPI", "Value"])
print(kpi_df.to_string(index=False))

# ─────────────────────────────────────────────
# 8. FINAL RANKED RECOMMENDATION TABLE
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 8: FINAL RANKED RECOMMENDATIONS (ALL PRODUCTS)")
print("=" * 60)

final_recs = (
    good.sort_values("Composite_Score", ascending=False)
    .groupby(["Product", "Region"])
    .first()
    .reset_index()
    .sort_values("Composite_Score", ascending=False)
)

final_recs["Rank"] = range(1, len(final_recs) + 1)

print(final_recs[[
    "Rank", "Product", "Region", "Current_Factory", "Alt_Factory",
    "LT_Reduction_Pct", "Dist_Reduction_km", "New_Est_Profit",
    "Composite_Score", "Confidence_Score"
]].head(25).to_string(index=False))

# ─────────────────────────────────────────────
# 9. SAVE ALL OUTPUTS
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 9: SAVING OUTPUTS")
print("=" * 60)

os.makedirs("/home/outputs", exist_ok=True)

good.to_csv("/home/outputs/scored_scenarios.csv", index=False)
top_per_product.to_csv("/home/outputs/top_recommendation_per_product.csv", index=False)
top5.to_csv("/home/outputs/top5_global_recommendations.csv", index=False)
final_recs.to_csv("/home/outputs/final_ranked_recommendations.csv", index=False)
kpi_df.to_csv("/home/outputs/kpi_summary.csv", index=False)
sim_df.to_csv("/home/outputs/simulation_with_risk.csv", index=False)

print("Saved: scored_scenarios.csv")
print("Saved: top_recommendation_per_product.csv")
print("Saved: top5_global_recommendations.csv")
print("Saved: final_ranked_recommendations.csv")
print("Saved: kpi_summary.csv")
print("Saved: simulation_with_risk.csv")
print("\n✅ Optimization & Recommendation Logic Complete!")

STEP 1: LOADING SIMULATION RESULTS
Simulation scenarios : 560
Baseline products    : 15
Route combinations   : 74

STEP 2: COMPOSITE OPTIMIZATION SCORING
Scored 214 beneficial scenarios
Composite Score range: 0.0582 – 0.8546

STEP 3: TOP FACTORY REASSIGNMENT RECOMMENDATIONS

🏆 TOP RECOMMENDATION PER PRODUCT:
                          Product   Current_Factory       Alt_Factory   Region      Ship_Mode  Baseline_LT  Predicted_LT  LT_Reduction_Pct  Dist_Reduction_km  Baseline_Profit  New_Est_Profit  Composite_Score  Confidence_Score
        Wonka Bar - Fudge Mallows     Lot's O' Nuts   Wicked Choccy's     Gulf Standard Class       394.09         33.57             91.48            1208.96            66.67           67.27           0.8546            0.5792
             Fizzy Lifting Drinks       Sugar Shack   Wicked Choccy's Atlantic    First Class       733.00        338.33             53.84            1660.17            60.00           60.83           0.6928            0.4096
            